In [1]:
# data processing
import pandas as pd
import imblearn
from pylab import *
%matplotlib inline
import pickle as pkl
import os
import rasterio as rio

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=UserWarning)
from training import *

# Load data

In [2]:
data_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/results/1_features/'
df = pd.read_csv(data_dir+'dataset_file_transformed.csv')
sites = df['site'].unique()

non_predictive_columns = ['x_pos', 'y_pos', 'site', 'class_certainty', 'veg_class']



# Train in-site model and save classifier as pickle object

In [3]:
save_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/results/2a_classifier/RF_is/'

feat_importance_dict = {}
cf_matrix_dict = {}
accuracies = []

#sites = ['CG1-8A', 'CG1-8B', 'CS-103A', 'CS-103B', 'CS-46A', 'CS-46B', 'CS-59B', 'CS-96B', 'CS117B', 'CS3A', 'CS3B', 'CS3C', 'F3-19B', 'F3-19C', 'F3-20A', 'F3-20B',
#         'F3-20C', 'F3-8A', 'F3-8B', 'F3-8C', 'SS33-1C', 'SS33-4A', 'SS33-9A', 'ZF20-11A', 'ZF20-35A', 'ZF20-4A', 'ZF20-59B', 'ZF20-6B', 'ZF20-8B', 'ZF20-9A', 
#         'ZF20-9C', 'ZF46-15A', 'ZF46-18B', 'ZF46-21B', 'ZF46-37A', 'ZF46-45A']

#sites = ['CG1-8A', 'CG1-8B', 'CS-103A', 'CS-103B', 'CS-46A', 'CS-46B', 'CS-59B', 'CS-96B', 'CS117B', 'CS3A', 'CS3B', 'F3-19B', 'F3-19C', 'F3-20A', 'F3-20B',
#         'F3-20C', 'F3-8A', 'F3-8B', 'F3-8C', 'SS33-1C', 'SS33-4A', 'SS33-9A', 'ZF20-11A', 'ZF20-35A', 'ZF20-4A', 'ZF20-59B', 'ZF20-6B', 'ZF20-8B', 'ZF20-9A', 
#         'ZF20-9C', 'ZF46-15A', 'ZF46-18B', 'ZF46-21B', 'ZF46-37A', 'ZF46-45A']

#sites = ['CS3C']
df_w = certainty_weight(df)

for site in sites:
    model, extra_metrics, _ = in_site_RF(df, site, extra_metrics=True, non_predictive_columns=non_predictive_columns)
    with open(save_dir+f'RF_{site}.pkl', 'wb') as file: 

        # A new file will be created 
        pkl.dump(model, file)
    
    # extra metrics
    feat_importance_dict[site] = extra_metrics['feature_importance']
    
    cf_matrix_dict[site] = extra_metrics['cf_matrix']
    
    train_acc = str(extra_metrics['train_accuracy'])
    test_acc = str(extra_metrics['test_accuracy'])
    accuracies.append(f'{site}, {train_acc}, {test_acc}')
    

# -----------------------------------------------------------------------------------------------------------------------------------------------------------------------
# saving model performance metrics (confusion matrix (cf), permutation importance, accuracies)
save_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/results/2b_classifier_performance/RF_is/'

with open(save_dir+f'RFis_cf_matrix.pkl', 'wb') as file:
    pkl.dump(cf_matrix_dict, file)
    
with open(save_dir+f'RFis_feat_importance.pkl', 'wb') as file:
        pkl.dump(feat_importance_dict, file)
        
with open(save_dir+'RFis_accuracies.txt', 'w') as outfile:
    outfile.write('site, train_acc, test_acc\n')
    outfile.write('\n'.join(str(i) for i in accuracies))
outfile.close()

Processing CG1-8A...
Processing CG1-8B...
Processing CS-103A...
Processing CS-103B...
Processing CS-46A...
Processing CS-46B...
Processing CS-59B...
Processing CS-96B...
Processing CS117B...
Processing CS3A...
Processing CS3B...
Processing CS3C...
Processing F3-19B...
Processing F3-19C...
Processing F3-20A...
Processing F3-20B...
Processing F3-20C...
Processing F3-8A...
Processing F3-8B...
Processing F3-8C...
Processing SS33-1C...
Processing SS33-4A...
Processing SS33-9A...
Processing ZF20-11A...
Processing ZF20-35A...
Processing ZF20-4A...
Processing ZF20-59B...
Processing ZF20-6B...
Processing ZF20-8B...
Processing ZF20-9A...
Processing ZF20-9C...
Processing ZF46-15A...
Processing ZF46-18B...
Processing ZF46-21B...
Processing ZF46-37A...
Processing ZF46-45A...


In [133]:
df_site = df_w[df_w['site'] == site]
c = Counter(df_site['veg_class'])
class_leastcommon = c.most_common()[:-2:-1][0][0]
class_leastcommon
df_class = df_site[df_site['veg_class'] == class_leastcommon]
df_class

,site,y_pos,x_pos,veg_class,R,G,B,chm,class_certainty,rc,gc,bc,rc/gc,rc+gc,ExG,ExR,ExB,ExGmExR,Ikaw,MGRVI,GLI,Y,L,z_score_Y,z_score_L
40702,CS3C,5003,1856,9,0.337255,0.333333,0.262745,1,2,0.361345,0.357143,0.281513,1.011765,0.718487,0.066667,0.138824,0.034510,-0.072157,0.124183,-0.011696,0.052632,0.088807,35.753541,-1.176045,-1.377991
40703,CS3C,5003,1857,9,0.325490,0.321569,0.250980,1,2,0.362445,0.358079,0.279476,1.012195,0.720524,0.066667,0.134118,0.029804,-0.067451,0.129252,-0.012121,0.054662,0.082438,34.485457,-1.212514,-1.457070
40704,CS3C,5004,1856,9,0.352941,0.349020,0.278431,1,2,0.360000,0.356000,0.284000,1.011236,0.716000,0.066667,0.145098,0.040784,-0.078431,0.118012,-0.011173,0.050147,0.097733,37.432536,-1.124933,-1.273287
40705,CS3C,5004,1857,9,0.337255,0.333333,0.262745,1,2,0.361345,0.357143,0.281513,1.011765,0.718487,0.066667,0.138824,0.034510,-0.072157,0.124183,-0.011696,0.052632,0.088807,35.753541,-1.176045,-1.377991
63509,CS3C,5003,1856,9,0.337255,0.333333,0.262745,1,2,0.361345,0.357143,0.281513,1.011765,0.718487,0.066667,0.138824,0.034510,-0.072157,0.124183,-0.011696,0.052632,0.088807,35.753541,-1.176045,-1.377991
63510,CS3C,5003,1857,9,0.325490,0.321569,0.250980,1,2,0.362445,0.358079,0.279476,1.012195,0.720524,0.066667,0.134118,0.029804,-0.067451,0.129252,-0.012121,0.054662,0.082438,34.485457,-1.212514,-1.457070
63511,CS3C,5004,1856,9,0.352941,0.349020,0.278431,1,2,0.360000,0.356000,0.284000,1.011236,0.716000,0.066667,0.145098,0.040784,-0.078431,0.118012,-0.011173,0.050147,0.097733,37.432536,-1.124933,-1.273287
63512,CS3C,5004,1857,9,0.337255,0.333333,0.262745,1,2,0.361345,0.357143,0.281513,1.011765,0.718487,0.066667,0.138824,0.034510,-0.072157,0.124183,-0.011696,0.052632,0.088807,35.753541,-1.176045,-1.377991
86316,CS3C,5003,1856,9,0.337255,0.333333,0.262745,1,2,0.361345,0.357143,0.281513,1.011765,0.718487,0.066667,0.138824,0.034510,-0.072157,0.124183,-0.011696,0.052632,0.088807,35.753541,-1.176045,-1.377991
86317,CS3C,5003,1857,9,0.325490,0.321569,0.250980,1,2,0.362445,0.358079,0.279476,1.012195,0.720524,0.066667,0.134118,0.029804,-0.067451,0.129252,-0.012121,0.054662,0.082438,34.485457,-1.212514,-1.457070


In [83]:
a = [1, 3, 5, 6, 9, 10, 14, 15, 56]
np.where(a == 10)

(array([], dtype=int64),)

In [28]:
sites = ['F3-20C']
for site in sites:
    df_site = df_w[df_w['site'] == site]
df_site

,site,y_pos,x_pos,veg_class,R,G,B,chm,class_certainty,rc,gc,bc,rc/gc,rc+gc,ExG,ExR,ExB,ExGmExR,Ikaw,MGRVI,GLI,Y,L,z_score_Y,z_score_L
3916,F3-20C,1016,4889,6,0.980392,0.996078,0.898039,3,1,0.341064,0.346521,0.312415,0.984252,0.687585,0.113725,0.376471,0.261176,-0.262745,0.043841,0.015872,0.029382,0.968648,98.774812,1.777897,1.407332
3917,F3-20C,1016,4890,6,0.984314,1.000000,0.905882,3,1,0.340570,0.345997,0.313433,0.984314,0.686567,0.109804,0.378039,0.268235,-0.268235,0.041494,0.015809,0.028226,0.977988,99.142528,1.812381,1.426140
3918,F3-20C,1017,4889,6,0.972549,0.984314,0.894118,3,1,0.341128,0.345254,0.313618,0.988048,0.686382,0.101961,0.377255,0.267451,-0.275294,0.042017,0.012024,0.026585,0.945523,97.854086,1.692516,1.360241
3919,F3-20C,1017,4890,6,0.992157,1.000000,0.913725,3,1,0.341430,0.344130,0.314440,0.992157,0.685560,0.094118,0.389020,0.279216,-0.294902,0.041152,0.007874,0.024096,0.982858,99.333367,1.830365,1.435900
3920,F3-20C,1104,4601,12,0.427451,0.509804,0.376471,1,1,0.325373,0.388060,0.286567,0.838462,0.713433,0.215686,0.088627,0.017255,0.127059,0.063415,0.174386,0.118280,0.200610,51.906111,-1.057810,-0.989816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115125,F3-20C,9278,1831,12,0.196078,0.247059,0.180392,1,3,0.314465,0.396226,0.289308,0.793651,0.710692,0.117647,0.027451,0.005490,0.090196,0.041667,0.227083,0.135135,0.044304,25.046077,-1.634915,-2.363601
120157,F3-20C,4267,895,10,0.756863,0.678431,0.690196,1,4,0.356089,0.319188,0.324723,1.115607,0.675277,-0.090196,0.381176,0.287843,-0.471373,0.046070,-0.108964,-0.032168,0.443592,72.467926,-0.160685,0.061839
120158,F3-20C,4267,896,10,0.741176,0.670588,0.674510,1,4,0.355263,0.321429,0.323308,1.105263,0.676692,-0.074510,0.367059,0.273725,-0.441569,0.047091,-0.099751,-0.027027,0.429232,71.502810,-0.213704,0.012477
120159,F3-20C,4268,895,10,0.749020,0.670588,0.678431,1,4,0.357009,0.319626,0.323364,1.116959,0.676636,-0.086275,0.378039,0.279216,-0.464314,0.049451,-0.110161,-0.031161,0.432193,71.703573,-0.202771,0.022746


In [38]:
from collections import Counter

site = 'F3-20A'
#check_samples = df['site'].isin([site])
check_samples = df[df['site'] == site]
c = min(Counter(check_samples['veg_class']).values())
print(c)

4


In [14]:
df[df['site'] == site]

,site,y_pos,x_pos,veg_class,R,G,B,chm,class_certainty,rc,gc,bc,rc/gc,rc+gc,ExG,ExR,ExB,ExGmExR,Ikaw,MGRVI,GLI,Y,L,z_score_Y,z_score_L
16264,F3-20A,499,6344,8,0.788235,0.768627,0.729412,1,2,0.344768,0.336192,0.319039,1.025510,0.680961,0.019608,0.334902,0.252549,-0.315294,0.038760,-0.025185,0.006418,0.554425,79.295489,0.832539,0.882473
16265,F3-20A,499,6345,8,0.835294,0.811765,0.772549,1,2,0.345219,0.335494,0.319287,1.028986,0.680713,0.015686,0.357647,0.269804,-0.341961,0.039024,-0.028566,0.004854,0.628030,83.338619,1.131354,1.067510
16266,F3-20A,500,6344,8,0.874510,0.854902,0.819608,1,2,0.343077,0.335385,0.321538,1.022936,0.678462,0.015686,0.369412,0.292549,-0.353725,0.032407,-0.022673,0.004608,0.704342,87.209399,1.441161,1.244659
16267,F3-20A,500,6345,8,0.847059,0.827451,0.784314,1,2,0.344498,0.336523,0.318979,1.023697,0.681021,0.023529,0.358431,0.270588,-0.334902,0.038462,-0.023416,0.007160,0.653576,84.667667,1.235064,1.128335
16268,F3-20A,585,8527,6,0.666667,0.741176,0.611765,2,3,0.330097,0.366990,0.302913,0.899471,0.697087,0.203922,0.192157,0.115294,0.011765,0.042945,0.105554,0.073864,0.473415,74.407706,0.503661,0.658780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17426,F3-20A,14803,2549,6,0.611765,0.678431,0.454902,2,3,0.350562,0.388764,0.260674,0.901734,0.739326,0.290196,0.178039,-0.041569,0.112157,0.147059,0.103068,0.119741,0.382160,68.179521,0.133189,0.373742
17427,F3-20A,14989,3331,13,0.517647,0.552941,0.529412,2,3,0.323529,0.345588,0.330882,0.936170,0.669118,0.058824,0.171765,0.188235,-0.112941,-0.011236,0.065862,0.027322,0.257046,57.755548,-0.374742,-0.103319
17428,F3-20A,14989,3332,13,0.560784,0.596078,0.580392,2,3,0.322799,0.343115,0.334086,0.940789,0.665914,0.050980,0.189020,0.216471,-0.138039,-0.017182,0.060960,0.021849,0.304342,62.027094,-0.182730,0.092172
17429,F3-20A,14990,3331,13,0.572549,0.603922,0.592157,2,3,0.323725,0.341463,0.334812,0.948052,0.665188,0.043137,0.197647,0.225098,-0.154510,-0.016835,0.053295,0.018182,0.314566,62.891170,-0.141226,0.131717


# Functions - code development
TODO: Outsource to aux file

In [ ]:
def certainty_weight(df):
    cert1_indices = df['class_certainty'].isin([1])
    cert2_indices = df['class_certainty'].isin([2])
    cert3_indices = df['class_certainty'].isin([3])
    cert4_indices = df['class_certainty'].isin([4])

    df_cert1 = pd.concat([df[cert1_indices].reset_index(drop=True)]*4, ignore_index=True)
    df_cert2 = pd.concat([df[cert2_indices].reset_index(drop=True)]*3, ignore_index=True)
    df_cert3 = pd.concat([df[cert3_indices].reset_index(drop=True)]*2, ignore_index=True)
    df_cert4 = pd.concat([df[cert4_indices].reset_index(drop=True)]*1, ignore_index=True)

    df_weighted = pd.concat([df_cert1, df_cert2, df_cert3, df_cert4], ignore_index=True)
    
    return df_weighted

In [2]:
def split_xy(df):
    feature_columns = [c for c in list(df.columns) if c not in non_predictive_columns]
    X = df[feature_columns]
    y = df['veg_class']
    return X, y


# Used for single site classifiers
def site_specific_RF(data, site, test_size=0.33, extra_metrics=False):
    print(f'Processing {site}...')
    df_site = df[df['site'] == site]
    df_reduced = df_site[df_site['class_certainty'] <= 2]
    X, y = split_xy(df_site)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

    # -------------------------------
    # oversample w synthetic minority oversampling to balance classes
    under = imblearn.over_sampling.SMOTE(sampling_strategy='not majority') # resamples all classes but majority
    X_res, y_res = under.fit_resample(X_train, y_train)

    # -----------------------------------------------------------------------------------------------------------
    # Train RF classifier
    clf=RandomForestClassifier(n_estimators=600, max_depth=6, n_jobs=12)
    clf.fit(X_res.values, y_res.values)

    # -----------------------------------------------------------------------------------------------------------
    # Prediction
    preds = clf.predict(X_test.values)

    #print(f'  Train accuracy: {clf.score(X_res, y_res)}')
    #print(f'  Test accuracy: {clf.score(X_test, y_test)}')

    if extra_metrics:
        # -----------------------------------------------------------------------------------------------------------
        # Inspect performance parameters: training and testing scores, confusion matrix, and feature importanceax1 = plt.figure(figsize=(5,5)).add_subplot(111)
        extra_metrics = {}

        extra_metrics['cf_matrix'] = confusion_matrix(y_test, preds,  labels = clf.classes_)
        #disp = ConfusionMatrixDisplay(confusion_matrix=cf_matrix, display_labels=clf.classes_)
        #disp.plot()
        #plt.title('Confusion Matrix')

        extra_metrics['feature_importance'] = pd.DataFrame(clf.feature_importances_, index=X_res.columns).sort_values(by=0, ascending=False)
        #print(feature_importance)
        
        extra_metrics['train_accuracy'] = clf.score(X_res, y_res)
        extra_metrics['test_accuracy'] = clf.score(X_test, y_test)
        
        return clf, extra_metrics, preds
    
    else:
        return clf, preds
    

In [52]:
#with open(data_dir+'cf_matrix.pkl', 'rb') as file:      
#    cf_matrix = pkl.load(file) 
#print(cf_matrix['CG1-8A'])

In [ ]:
model_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/RF_classifers/'
with open(model_dir+'RF_CS-103A.pkl', 'rb') as file:      
    model = pkl.load(file) 
#print(cf_matrix['CG1-8A'])

# Classify site

In [53]:
site = 'CS-103A'
run_name = f'{site}_01'

# location of raw lichen data
data_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/2_labelling/hp'

# place where we stored chunked feature data
run_dir = f'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/runs/{run_name}'

Processing block 0-0
Processing block 0-1
Processing block 0-2
Processing block 0-3
Processing block 1-0
Processing block 1-1
Processing block 1-2
Processing block 1-3
Processing block 2-0
Processing block 2-1
Processing block 2-2
Processing block 2-3
Processing block 3-0
Processing block 3-1
Processing block 3-2
Processing block 3-3


In [60]:
#out_file = '/network/scratch/m/matthew.fortier/CS-103A-classified.tif'
out_file = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/runs/CS-103A-classified.tif'

meta.update(count=1)

# Write a new .tif file using the metadata from the original file
with rio.open(out_file, 'w', **meta) as dst:
    dst.write(output, 1)